In [1]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# data manipulation
import pandas as pd
import numpy as np

#visualizatio
import seaborn as sns
import matplotlib.pyplot as plt

# warning
import warnings
warnings.filterwarnings("ignore")

# display settings
pd.set_option('display.max_columns', None)


In [3]:
# load dataset
df = pd.read_csv('/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/raw/PS_20174392719_1491204439457_log.csv')
df.head(10)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.0,0.00,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.0,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.0,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.0,0.00,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.0,0.00,0,0
5,1,PAYMENT,7817.71,C90045638,53860.00,46042.29,M573487274,0.0,0.00,0,0
6,1,PAYMENT,7107.77,C154988899,183195.00,176087.23,M408069119,0.0,0.00,0,0
7,1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.00,0,0
8,1,PAYMENT,4024.36,C1265012928,2671.00,0.00,M1176932104,0.0,0.00,0,0
9,1,DEBIT,5337.77,C712410124,41720.00,36382.23,C195600860,41898.0,40348.79,0,0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [5]:
# check current memory usage
print("Memory usage before optimization:")
print(df.memory_usage(deep=True).sum() / 1024**2, "MB")


# optimize integer columns
df["step"] = df["step"].astype("int16")
df["isFraud"] = df["isFraud"].astype("int8")
df["isFlaggedFraud"] = df["isFlaggedFraud"].astype("int8")


# optimize float columns
float_cols = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

for col in float_cols:
    df[col] = pd.to_numeric(df[col], downcast="float")


# convert categorical/object columns
df["type"] = df["type"].astype("category")


# check optimized memory usage
print("\nMemory usage after optimization:")
print(df.memory_usage(deep=True).sum() / 1024**2, "MB")

Memory usage before optimization:
1452.5654621124268 MB

Memory usage after optimization:
994.9128046035767 MB


In [6]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='object')

In [7]:
df[['nameOrig', 'nameDest']].head()

,nameOrig,nameDest
0,C1231006815,M1979787155
1,C1666544295,M2044282225
2,C1305486145,C553264065
3,C840083671,C38997010
4,C2048537720,M1230701703


In [8]:
df['nameOrig'].nunique()

6353307

In [9]:
df['nameDest'].nunique()

2722362

In [10]:
# extract account type prefixes
df["origin_type"] = df["nameOrig"].str[0]

df["destination_type"] = df["nameDest"].str[0]

# preview
df[["nameOrig", "origin_type", "nameDest", "destination_type"]].head()

,nameOrig,origin_type,nameDest,destination_type
0,C1231006815,C,M1979787155,M
1,C1666544295,C,M2044282225,M
2,C1305486145,C,C553264065,C
3,C840083671,C,C38997010,C
4,C2048537720,C,M1230701703,M


In [11]:
# drop raw identifier columns
df.drop(columns=["nameOrig", "nameDest"], inplace=True)

# confirm remaining columns
df.columns

Index(['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
       'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud',
       'origin_type', 'destination_type'],
      dtype='object')

In [12]:
from sklearn.model_selection import train_test_split
# separate features and target temporarily for sampling
x_temp = df.drop(columns=["isFraud"], axis=1)
y_temp = df["isFraud"]

# stratified sampling
X_sample, _, y_sample, _ = train_test_split(x_temp, y_temp, stratify=y_temp, train_size=1000000, random_state=42)

# combine dataset
sample_df = X_sample.copy()
sample_df["isFraud"] = y_sample

# check shape
print(sample_df.shape)

# check fraud distribution
print(sample_df["isFraud"].value_counts(normalize=True))

(1000000, 11)
isFraud
0    0.998709
1    0.001291
Name: proportion, dtype: float64


In [13]:
df['isFraud'].value_counts(normalize=True)

,proportion
isFraud,
0,0.998709
1,0.001291


In [14]:
# delete unused large objects
del x_temp
del y_temp
del X_sample
del y_sample
del df
# garbage collection
import gc
gc.collect()

0

In [15]:
sample_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000000 entries, 3640776 to 495640
Data columns (total 11 columns):
 #   Column            Non-Null Count    Dtype   
---  ------            --------------    -----   
 0   step              1000000 non-null  int16   
 1   type              1000000 non-null  category
 2   amount            1000000 non-null  float64 
 3   oldbalanceOrg     1000000 non-null  float64 
 4   newbalanceOrig    1000000 non-null  float64 
 5   oldbalanceDest    1000000 non-null  float64 
 6   newbalanceDest    1000000 non-null  float64 
 7   isFlaggedFraud    1000000 non-null  int8    
 8   origin_type       1000000 non-null  object  
 9   destination_type  1000000 non-null  object  
 10  isFraud           1000000 non-null  int8    
dtypes: category(1), float64(5), int16(1), int8(2), object(2)
memory usage: 65.8+ MB


In [16]:
# sender balance change
sample_df["sender_balance_change"] = sample_df["oldbalanceOrg"] - sample_df["newbalanceOrig"]

# Receiver balance change
sample_df["destination_balance_change"] = sample_df["newbalanceDest"] - sample_df["oldbalanceDest"]

# preview
sample_df[["oldbalanceOrg", "newbalanceOrig", "sender_balance_change",
           "oldbalanceDest", "newbalanceDest", "destination_balance_change"]].head()

,oldbalanceOrg,newbalanceOrig,sender_balance_change,oldbalanceDest,newbalanceDest,destination_balance_change
3640776,0.0,0.00,0.00,0.00,0.00,0.00
3127218,75109.0,74448.38,660.62,1286830.01,1287490.64,660.63
4752380,132304.0,174955.87,-42651.87,0.00,0.00,0.00
6356543,39438.0,406947.16,-367509.16,0.00,0.00,0.00
3902308,40243.0,0.00,40243.00,2068662.71,2184335.48,115672.77


In [17]:
sample_df.reset_index(drop=True, inplace=True)

In [18]:
sample_df.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFlaggedFraud,origin_type,destination_type,isFraud,sender_balance_change,destination_balance_change
0,275,PAYMENT,21226.31,0.0,0.00,0.00,0.00,0,C,M,0,0.00,0.00
1,236,CASH_OUT,660.62,75109.0,74448.38,1286830.01,1287490.64,0,C,C,0,660.62,660.63
2,333,CASH_IN,42651.87,132304.0,174955.87,0.00,0.00,0,C,C,0,-42651.87,0.00
3,710,CASH_IN,367509.16,39438.0,406947.16,0.00,0.00,0,C,C,0,-367509.16,0.00
4,284,CASH_OUT,115672.77,40243.0,0.00,2068662.71,2184335.48,0,C,C,0,40243.00,115672.77


In [19]:
# sender balance inconsistency
sample_df["origin_balance_error"] = (sample_df["sender_balance_change"] - sample_df["amount"])

# Receiver balance inconsistency
sample_df["destination_balance_error"] = (sample_df["destination_balance_change"] - sample_df["amount"])

# preview
sample_df[
    [
        "amount",
        "sender_balance_change",
        "origin_balance_error",
        "destination_balance_change",
        "destination_balance_error"
    ]
].head()

,amount,sender_balance_change,origin_balance_error,destination_balance_change,destination_balance_error
0,21226.31,0.00,-2.122631e+04,0.00,-2.122631e+04
1,660.62,660.62,-4.661160e-12,660.63,1.000000e-02
2,42651.87,-42651.87,-8.530374e+04,0.00,-4.265187e+04
3,367509.16,-367509.16,-7.350183e+05,0.00,-3.675092e+05
4,115672.77,40243.00,-7.542977e+04,115672.77,1.455192e-11


In [20]:
sample_df[
    [
        "amount",
        "sender_balance_change",
        "destination_balance_change",
        "origin_balance_error",
        "destination_balance_error"
    ]
].describe()

,amount,sender_balance_change,destination_balance_change,origin_balance_error,destination_balance_error
count,1.000000e+06,1.000000e+06,1.000000e+06,1.000000e+06,1.000000e+06
mean,1.802541e+05,-2.113933e+04,1.242200e+05,-2.013934e+05,-5.603402e+04
std,6.094990e+05,1.541404e+05,8.123285e+05,6.104269e+05,4.420144e+05
min,0.000000e+00,-1.915268e+06,-5.353304e+06,-6.988673e+07,-1.000000e+07
25%,1.341844e+04,0.000000e+00,0.000000e+00,-2.498744e+05,-2.954647e+04
50%,7.478222e+04,0.000000e+00,0.000000e+00,-6.864275e+04,-3.550165e+03
75%,2.088243e+05,1.013500e+04,1.487626e+05,-2.958697e+03,0.000000e+00
max,6.988673e+07,1.000000e+07,8.270459e+07,1.000000e-02,6.156804e+07


In [21]:
# inspect category and object column
sample_df.select_dtypes(include=['category', 'object']).nunique()

,0
type,5
origin_type,1
destination_type,2


In [22]:
sample_df.select_dtypes(include=['category', 'object']).head()

,type,origin_type,destination_type
0,PAYMENT,C,M
1,CASH_OUT,C,C
2,CASH_IN,C,C
3,CASH_IN,C,C
4,CASH_OUT,C,C


In [23]:
# drop origin_type column
sample_df.drop(columns=["origin_type"], inplace=True)

# verify
sample_df.select_dtypes(include=['category', 'object']).nunique()
#

,0
type,5
destination_type,2


In [24]:
# one-hot encode categorical columns
sample_df = pd.get_dummies(sample_df,
                           drop_first=True,
                           columns=['type', 'destination_type'],
                           dtype='int8')

# preview
sample_df.head()
#

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFlaggedFraud,isFraud,sender_balance_change,destination_balance_change,origin_balance_error,destination_balance_error,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,destination_type_M
0,275,21226.31,0.0,0.00,0.00,0.00,0,0,0.00,0.00,-2.122631e+04,-2.122631e+04,0,0,1,0,1
1,236,660.62,75109.0,74448.38,1286830.01,1287490.64,0,0,660.62,660.63,-4.661160e-12,1.000000e-02,1,0,0,0,0
2,333,42651.87,132304.0,174955.87,0.00,0.00,0,0,-42651.87,0.00,-8.530374e+04,-4.265187e+04,0,0,0,0,0
3,710,367509.16,39438.0,406947.16,0.00,0.00,0,0,-367509.16,0.00,-7.350183e+05,-3.675092e+05,0,0,0,0,0
4,284,115672.77,40243.0,0.00,2068662.71,2184335.48,0,0,40243.00,115672.77,-7.542977e+04,1.455192e-11,1,0,0,0,0


In [25]:
# separate features and target
from sklearn.model_selection import train_test_split

X = sample_df.drop(columns=["isFraud"], axis=1)
y = sample_df["isFraud"]

# train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# check shapes
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

# check fraud distribution
print(f"y_train fraud distribution: {y_train.value_counts(normalize=True)}")
print(f"\ny_test fraud distribution: {y_test.value_counts(normalize=True)}")

X_train shape: (800000, 16)
X_test shape: (200000, 16)
y_train fraud distribution: isFraud
0    0.998709
1    0.001291
Name: proportion, dtype: float64

y_test fraud distribution: isFraud
0    0.99871
1    0.00129
Name: proportion, dtype: float64


In [26]:
# drop potential leakage feature
X_train = X_train.drop(columns=["isFlaggedFraud"])
X_test = X_test.drop(columns=["isFlaggedFraud"])

# verify
X_train.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,sender_balance_change,destination_balance_change,origin_balance_error,destination_balance_error,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,destination_type_M
370579,184,40653.79,10399.0,0.00,0.00,0.00,10399.00,0.00,-3.025479e+04,-40653.79,0,0,1,0,1
973226,324,134239.52,4600966.4,4735205.93,1145839.79,1078464.75,-134239.53,-67375.04,-2.684790e+05,-201614.56,0,0,0,0,0
413737,547,8051.53,40999.0,32947.47,0.00,0.00,8051.53,0.00,-9.094947e-13,-8051.53,0,0,1,0,1
852124,15,161210.10,0.0,0.00,341586.49,502796.60,0.00,161210.11,-1.612101e+05,0.01,1,0,0,0,0
860933,250,51807.07,597469.0,545661.93,0.00,0.00,51807.07,0.00,-5.093170e-11,-51807.07,0,0,1,0,1


In [27]:
sample_df[['destination_balance_error', 'origin_balance_error', 'step']].describe()

,destination_balance_error,origin_balance_error,step
count,1.000000e+06,1.000000e+06,1000000.000000
mean,-5.603402e+04,-2.013934e+05,243.545358
std,4.420144e+05,6.104269e+05,142.429759
min,-1.000000e+07,-6.988673e+07,1.000000
25%,-2.954647e+04,-2.498744e+05,156.000000
50%,-3.550165e+03,-6.864275e+04,240.000000
75%,0.000000e+00,-2.958697e+03,335.000000
max,6.156804e+07,1.000000e-02,743.000000


In [28]:
# continuous numerical features for transformation
continuous_features = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "sender_balance_change",
    "destination_balance_change",
    "origin_balance_error",
    "destination_balance_error"
]

# remaining features
binary_features = [
    col for col in X_train.columns
    if col not in continuous_features
]

print("Continuous Features:")
print(continuous_features)

print("\nRemaining Features:")
print(binary_features)

Continuous Features:
['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'sender_balance_change', 'destination_balance_change', 'origin_balance_error', 'destination_balance_error']

Remaining Features:
['step', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER', 'destination_type_M']


In [29]:
from sklearn.preprocessing import PowerTransformer

# initialize transformer
pt = PowerTransformer(method="yeo-johnson")

# fit and transform training data
X_train_continuous = pt.fit_transform(X_train[continuous_features])

# transform test data
X_test_continuous = pt.transform(X_test[continuous_features])

In [30]:
# convert transformed training data to dataframe
X_train_continuous = pd.DataFrame(
    X_train_continuous,
    columns=continuous_features,
    index=X_train.index
)

# convert transformed test data to dataframe
X_test_continuous = pd.DataFrame(
    X_test_continuous,
    columns=continuous_features,
    index=X_test.index
)

# preview
X_train_continuous.head()

,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,sender_balance_change,destination_balance_change,origin_balance_error,destination_balance_error
370579,-0.229770,0.238214,-0.864429,-1.135035,-1.225006,0.241406,0.051768,0.024323,0.139169
973226,0.475548,1.529514,1.399901,0.926833,0.828952,-0.805927,-0.392042,-0.810921,-0.552660
413737,-1.047527,0.505615,0.947502,-1.135035,-1.225006,0.227104,0.051768,1.588712,0.269489
852124,0.592196,-1.267432,-0.864429,0.700084,0.658459,0.177532,0.254559,-0.587791,0.298892
860933,-0.094120,1.066073,1.225339,-1.135035,-1.225006,0.491060,0.051768,1.588712,0.093108


In [31]:
# untouched features
X_train_binary = X_train[binary_features]
X_test_binary = X_test[binary_features]

# combine transformed + untouched features
X_train_linear = pd.concat(
    [X_train_continuous, X_train_binary],
    axis=1
)

X_test_linear = pd.concat(
    [X_test_continuous, X_test_binary],
    axis=1
)

# check shape
print("X_train_linear shape:", X_train_linear.shape)
print("X_test_linear shape:", X_test_linear.shape)

# preview
X_train_linear.head()

X_train_linear shape: (800000, 15)
X_test_linear shape: (200000, 15)


,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,sender_balance_change,destination_balance_change,origin_balance_error,destination_balance_error,step,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,destination_type_M
370579,-0.229770,0.238214,-0.864429,-1.135035,-1.225006,0.241406,0.051768,0.024323,0.139169,184,0,0,1,0,1
973226,0.475548,1.529514,1.399901,0.926833,0.828952,-0.805927,-0.392042,-0.810921,-0.552660,324,0,0,0,0,0
413737,-1.047527,0.505615,0.947502,-1.135035,-1.225006,0.227104,0.051768,1.588712,0.269489,547,0,0,1,0,1
852124,0.592196,-1.267432,-0.864429,0.700084,0.658459,0.177532,0.254559,-0.587791,0.298892,15,1,0,0,0,0
860933,-0.094120,1.066073,1.225339,-1.135035,-1.225006,0.491060,0.051768,1.588712,0.093108,250,0,0,1,0,1


In [32]:
from imblearn.over_sampling import SMOTE
# initialize SMOTE
smote = SMOTE(sampling_strategy=0.2,random_state=42)

# apply SMOTE ONLY on training data
X_train_linear_smote, y_train_linear_smote = smote.fit_resample(
    X_train_linear,
    y_train
)

# check class distribution
print("Original Training Distribution:")
print(y_train.value_counts())

print("\nSMOTE Training Distribution:")
print(y_train_linear_smote.value_counts())

Original Training Distribution:
isFraud
0    798967
1      1033
Name: count, dtype: int64

SMOTE Training Distribution:
isFraud
0    798967
1    159793
Name: count, dtype: int64


In [33]:
# raw datasets for tree/boosted models
X_train_tree = X_train.copy()
X_test_tree = X_test.copy()

# check shapes
print("X_train_tree shape:", X_train_tree.shape)
print("X_test_tree shape:", X_test_tree.shape)

# preview
X_train_tree.head()

X_train_tree shape: (800000, 15)
X_test_tree shape: (200000, 15)


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,sender_balance_change,destination_balance_change,origin_balance_error,destination_balance_error,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,destination_type_M
370579,184,40653.79,10399.0,0.00,0.00,0.00,10399.00,0.00,-3.025479e+04,-40653.79,0,0,1,0,1
973226,324,134239.52,4600966.4,4735205.93,1145839.79,1078464.75,-134239.53,-67375.04,-2.684790e+05,-201614.56,0,0,0,0,0
413737,547,8051.53,40999.0,32947.47,0.00,0.00,8051.53,0.00,-9.094947e-13,-8051.53,0,0,1,0,1
852124,15,161210.10,0.0,0.00,341586.49,502796.60,0.00,161210.11,-1.612101e+05,0.01,1,0,0,0,0
860933,250,51807.07,597469.0,545661.93,0.00,0.00,51807.07,0.00,-5.093170e-11,-51807.07,0,0,1,0,1


In [34]:
# initialize SMOTE
smote_tree = SMOTE(sampling_strategy=0.2,random_state=42)

# apply SMOTE on raw tree training data
X_train_tree_smote, y_train_tree_smote = smote_tree.fit_resample(
    X_train_tree,
    y_train
)

# check class distribution
print("Original Training Distribution:")
print(y_train.value_counts())

print("\nTree SMOTE Distribution:")
print(y_train_tree_smote.value_counts())

Original Training Distribution:
isFraud
0    798967
1      1033
Name: count, dtype: int64

Tree SMOTE Distribution:
isFraud
0    798967
1    159793
Name: count, dtype: int64


In [35]:
import joblib
import os

# save linear datasets
joblib.dump(X_train_linear, "/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/X_train_linear.pkl")
joblib.dump(X_train_linear_smote, "/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/X_train_linear_smote.pkl")
joblib.dump(X_test_linear, "/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/X_test_linear.pkl")

# save tree datasets
joblib.dump(X_train_tree, "/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/X_train_tree.pkl")
joblib.dump(X_train_tree_smote, "/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/X_train_tree_smote.pkl")
joblib.dump(X_test_tree, "/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/X_test_tree.pkl")

# save labels
joblib.dump(y_train, "/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/y_train.pkl")
joblib.dump(y_train_linear_smote, "/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/y_train_linear_smote.pkl")
joblib.dump(y_train_tree_smote, "/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/y_train_tree_smote.pkl")
joblib.dump(y_test, "/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/data/processed/y_test.pkl")

print("All processed datasets saved successfully.")

All processed datasets saved successfully.


In [36]:
# save fitted PowerTransformer
joblib.dump(pt, "/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/model/power_transformer.pkl")

['/content/drive/MyDrive/ML_Portfolio/fraud-detection-ml/model/power_transformer.pkl']

## Phase Summary

In this phase, the dataset was prepared for machine learning modeling through feature engineering, preprocessing, encoding, and class imbalance handling.

Key behavioral fraud-related features were engineered from account balance movements and transaction inconsistencies. Categorical variables were encoded using one-hot encoding, while low-information and leakage-prone features were removed to improve modeling reliability.

To support different model families, separate preprocessing strategies were implemented:

* Linear and distance-based models used PowerTransformer preprocessing and SMOTE oversampling.
* Tree and boosting models retained raw engineered features with optional SMOTE oversampling.

The final outputs of this phase included:

* transformed datasets for linear models,
* raw datasets for tree-based models,
* oversampled training datasets,
* and untouched test datasets for realistic evaluation.

This preprocessing framework ensures fair model comparison, minimizes data leakage, and prepares the project for systematic model training and evaluation.
